#RAG Agent for Document Retrieval using AWS Strands Agent and Vector Database - ChromaDB

##RAG - solves the problem of LLM - that its knowledge is outdated.

Directly provide what ever knowledge - new knowledge LLM doesnt know about or special domain of knowledge that we want the LLM to pay more attention to so we get more relevant and less hallucinated responses.

##Connect to GPU and start running
##Key packages
### - strands-agents - agent framework just like LangChain and LangGraph. You can use LC and LG too as your agents.
### - Vector Database - we use ChromaDB - very begineer friendly , open src and user friendly, can easily set up.
### - For document parsing and Chunking - I am using docling and chonkie - used to parser docs in the right format and use chonkie to chunk it according to the mentioned delimiters.

In [ ]:
!pip install strands-agents chromadb docling chonkie

We will use ChromaDB to store out document embeddings and Retrieve them based on semantic similarity.

## Setup the ChromaDB to set up everything.

Now, we will load the Qwen3-Embedding-0.6B model using the embedding_functions - It is a utility module built into ChromaDB that gives you ready-made connectors to popular embedding models. Instead of you writing the code to load a model, feed text into it, and get vectors back — ChromaDB wraps all of that into one simple object.

embedding_functions is the adaptor that connects them without you having to wire anything yourself.

SentenceTransformerEmbeddingFunction — a pre-built connector for any model from the SentenceTransformers library. Qwen3-Embedding is one such model.
model_name="Qwen/Qwen3-Embedding-0.6B" — tells it which specific embedding model to download and use. This is the model that converts text into vectors.
device="cuda" — tells it to run on your GPU instead of CPU. Faster computation.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# I have created my own document chunks - list of strings
doc_chunks = [
    "Artificial intelligence has a long history, dating back to the mid-20th century. Early pioneers like Alan Turing and John McCarthy laid the groundwork for the field. The development of machine learning in recent decades has led to rapid advancements.",
    "Large language models (LLMs) are a type of AI model trained on massive amounts of text data. They can generate human-like text, translate languages, and answer questions in an informative way. GPT-3, developed by OpenAI, is a well-known example.",
    "Vector databases are essential for building modern AI applications. They store and query high-dimensional vectors, such as those generated by embedding models. This enables semantic search and other advanced features in applications like RAG agents."
]

embedding_mod=embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="Qwen/Qwen3-Embedding-0.6B",  # Which embedding model to use — Qwen3 here
    device="cuda" # Run on GPU not CPU
    )
# 2. Pass this function when retrieving your collection
client = chromadb.PersistentClient(path="./rag_vec_db")  # This creates a ChromaDB client that saves
                                                         # everything to disk at the folder path ./rag_vec_db.
                                                        # just tells it which folder to save the database files into. If the folder doesn't exist, ChromaDB creates it automatically.
#The word Persistent is the key here. There are two types of ChromaDB clients:

#EphemeralClient — stores everything in RAM. When your Colab session ends, everything is gone forever.
#PersistentClient — stores everything on disk. When your Colab session restarts, your database is still there.

collection = client.get_or_create_collection(
    name="rag_collection",
    embedding_function=embedding_mod,
    metadata={"hnsw:space": "cosine"} # Change the similarity here to cosine distance
)

collection.add(
    documents=doc_chunks,
    ids=[f"id_{i}" for i in range(len(doc_chunks))]
)

print(f"Successfully Created chroma vector database!")



Now you have set up the local database. Once this is done - you can just Run the Queries.


This is the retrieval step — the R in RAG. This is where you actually search the vector database with a question and get the most relevant chunks back.
Let me break every line down.

top_n = 2 means - I want the top 2 most relevant chunks returned. You could set this to 3, 5, 10 .. whatever you want. In production RAG systems this is usually 3 to 5.

In [ ]:
top_n = 2 # Want to return top 2
results = collection.query(
    query_texts=["What is vector database?"],
    include=["documents", "distances", "embeddings"],
    n_results=top_n
) # 'ids', 'documents', 'metadatas', and 'embeddings' are returned


print("\n".join(results['documents'][0]))
print("Distance: ", results['distances'][0])  # lower the distance more is the similarity and more relevant it is
print("Embedding dim: ", len(results['embeddings'][0][0]))

Pre-built tools — already wrapped for you
Tavily comes as a pre-built integration. Someone at AWS or the Tavily team already wrote the tool wrapper. You just import it and hand it directly to the agent.

Custom tools — you build it yourself
When you write your own function — like this RAG tool — Strands has no idea it is supposed to be a tool. It just looks like a regular Python function. The @tool decorator is what tells Strands:

"Hey, this function is a tool. Make it available for the agent to call."


In [ ]:
from strands import tool

@tool
def rag_tool(query: str) -> str:
    """Retrieve the relevant document according to the query from vector database."""
    # Query the Qwen3 collection to find the most relevant document chunks
    results = collection.query(
        query_texts=[query],
        n_results=2,
    )

    # Combine the retrieved chunks into a single context
    retrieved_context = "\n\n".join(results['documents'][0])

    return retrieved_context

print("RAG tool created successfully.")

##After we create our tools then we need our LLM.

### Amazon Bedrock is a fully managed AWS service that provides developers with a single, unified API to access high-performing foundation models from top AI companies (like Anthropic, Meta, and Amazon). It enables seamless building and scaling of generative AI applications without needing to manage any underlying infrastructure.

### To use Bedrock you need to prove to AWS that you are an authorised user. AWS does this through three values:

### AWS_ACCESS_KEY_ID — your username for AWS services. Like a user ID.

### AWS_SECRET_ACCESS_KEY — your password for AWS services. Secret, never shared.

### AWS_DEFAULT_REGION — which AWS data centre to use. For example "us-east-1" means AWS servers in North Virginia.



In [ ]:
import os
from google.colab import userdata

# Load AWS credentials from Colab secrets
# This is a Colab-specific feature. Instead of hardcoding your secret keys directly in your notebook
os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = userdata.get("AWS_DEFAULT_REGION")


### AWS has data centres all over the world. The us. prefix means use the US inference profile — AWS automatically routes your request to the best available US data centre. Without this prefix you would have to specify an exact region. The us. prefix makes it more flexible and reliable.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-6"  # Note the "us." prefix
)
agent = Agent(model=model, callback_handler=None,
              tools=[rag_tool],
              system_prompt="""You are a helpful assistant that uses the RAG tool to answer
              questions based on the provided documents. Always cite the retrieved
              information in your responses.""")

print("Chroma-powered RAG agent created successfully.")

## Invoke the Agent

### Let's test our new agent to see how the Qwen3 embeddings improve the retrieval and response quality.

In [ ]:
query1 = "Is AI helping skill formation or not?"
print(f"Query: {query1}")
response1 = str(agent(query1))
print(f"Agent Response: {response1}")

## We have successfully built a RAG agent using the Strands Agent framework and Chroma

#Chunking a long document

### 1. code for the user to upload a PDF file from their local machine to the Colab environment using google.colab.files

In [ ]:
from google.colab import files

print("Please upload your PDF file")
uploaded = files.upload()

input_doc = list(uploaded.keys())[0]
print(input_doc)

### 2. docling to convert the pdf into a markdown format that best perserves the sections and layout of documents.

### 3. chonkie to chunk the documents with custom delimiters

### can also try using the HybridChunker directly provided by docling

In [ ]:
from typing import Text
from chonkie import RecursiveChunker, RecursiveRules, RecursiveLevel
from docling.document_converter import DocumentConverter

def chunk_pdf_by_chapters(pdf_path: str, chunk_size = 2048) -> list:
    """
    Chunks a PDF by chapters and sections using a Markdown-based approach and Chonkie.

    Args:
        pdf_path: URL or The file path to the PDF document.

    Returns:
        A list of doc chunks.
    """

    # 1. Convert PDF to Markdown using docling (preserves structure)
    try:
        converter = DocumentConverter()
        result = converter.convert(pdf_path)
        pdf_markdown = result.document.export_to_markdown()
    except Exception as e:
        print(f"Error converting PDF to Markdown: {e}")
        return []

    # 2. Initialize Chonkie for advanced text splitting
    custom_rules = RecursiveRules(
      levels=[
          # Level 1: Split by markdown-style headers
          RecursiveLevel(delimiters=["#", "##", "###", "####", "#####"], include_delim="next"),

          # Level 2: Split by paragraph breaks (double newlines)
          RecursiveLevel(delimiters=["\n\n"]),

          # Level 3: Split by standard sentence punctuation
          RecursiveLevel(delimiters=[". ", "! ", "? "], include_delim="prev"),

          # Level 4: Fallback to split at whitespace if chunk is still too large
          RecursiveLevel(whitespace=True)])

    chunker = RecursiveChunker(
        rules=custom_rules,
        chunk_size=chunk_size
    )

    # 3. Chunk the markdown text using Chonkie
    raw_chunks = chunker.chunk(pdf_markdown)

    # 4. Convert raw string chunks into LangChain Document objects
    chunks = [chunk.text for chunk in raw_chunks]

    return chunks

In [ ]:
chunks = chunk_pdf_by_chapters(input_doc, 2048)
print("number of chunks", len(chunks))
print(chunks[0])

In [ ]:
print(chunks[2]) # 3rd Chunk

### You want to pass the 68 chunks created into the collection

#### This takes a lot fo time to execute - coz of number of chunks and also out context window size that we have set.

In [ ]:
collection = client.get_or_create_collection(
    name="AI_collection",
    embedding_function=embedding_mod,
)

collection.add(
    documents=chunks,
    ids=[f"id_{i}" for i in range(len(chunks))]
)

print(f"Successfully create chroma vector database collection!")

## How to parse the Figues in the document.

### Using docling
#### Crops the pdf for images

In [ ]:
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
import os
import shutil

# 1. Configure pipeline to capture images
pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2.0  # Higher quality

# 2. Assign options specifically to the PDF format
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

result = converter.convert(input_doc)

# 3. Create output folder. Delete the folder if existing
folder_name = "figures"
if os.path.exists(folder_name):
    print(f"Deleting existing folder: {folder_name}")
    shutil.rmtree(folder_name)
    print(f"Successfully deleted {folder_name}")

# 4. Create the new folder for output
else:
    os.makedirs(folder_name)
    print(f"Successfully created new folder: {folder_name}")

# 5. Use get_image() and Pillow to save
for i, element in enumerate(result.document.pictures):
    print("Extracting figure: ", i)
    # This retrieves the actual PIL image object
    pil_img = element.get_image(result.document)

    if pil_img:
        # Format i as a three-digit string
        pil_img.save(f"./{folder_name}/figure_{i:03d}.png", "PNG")